# DRI-24 GGUF Export For llama.cpp

This notebook exports the DRI-23 quantized Qwen2-VL run #4 checkpoint to llama.cpp artifacts. It tries the uploaded LLM Compressor quantized Hugging Face checkpoint first, then writes:

- a text-model GGUF intermediate
- a deployable `Q4_K_M` text GGUF via `llama-quantize`
- a separate `mmproj` GGUF for Qwen2-VL vision input
- a JSON export report with sizes and validation output

Current llama.cpp conversion does not accept `q4_k_m` directly in `convert_hf_to_gguf.py`; it only supports `f32`, `f16`, `bf16`, `q8_0`, `tq1_0`, `tq2_0`, and `auto`. So this notebook converts to `f16` first, then runs `llama-quantize ... Q4_K_M`.

In [ ]:
from pathlib import Path

PROJECT_REPO = "https://github.com/ShivamSinghNow/Drishti.git"
BRANCH = "codex/dri24-gguf-export"
PROJECT_DIR = Path("/content/Drishti")

HF_QUANT_REPO_ID = "ShivSingh123/drishti-qwen2vl-run4-llmcompressor-gptq-int4"
HF_GGUF_REPO_ID = "ShivSingh123/drishti-qwen2vl-run4-gguf"

QUANT_DIR = Path("outputs/dri23-run4-llmcompressor-gptq-int4")
MERGED_DIR = Path("outputs/dri23-run4-merged-fp16")
GGUF_DIR = Path("outputs/dri24-gguf")
LLAMA_CPP_DIR = Path("external/llama.cpp")

In [ ]:
import os
import subprocess
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def run(command: list[str], *, cwd: Path | None = None, capture: bool = False) -> subprocess.CompletedProcess | None:
    print("\n$ " + " ".join(command))
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        text=True,
        capture_output=capture,
    )
    if capture:
        print(result.stdout)
        print(result.stderr)
    result.check_returncode()
    return result if capture else None

In [ ]:
if not PROJECT_DIR.exists():
    run(["git", "clone", PROJECT_REPO, str(PROJECT_DIR)])

run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
run(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)

%cd /content/Drishti
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], capture=True)
run(["apt-get", "update"], capture=True)
run(["apt-get", "install", "-y", "cmake", "ninja-build", "build-essential", "libcurl4-openssl-dev"], capture=True)

In [ ]:
import os
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata
except Exception:
    userdata = None

if userdata is not None:
    hf_token = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN")
    kaggle_username = userdata.get("KAGGLE_USERNAME")
    kaggle_key = userdata.get("KAGGLE_KEY")
else:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    kaggle_username = os.environ.get("KAGGLE_USERNAME")
    kaggle_key = os.environ.get("KAGGLE_KEY")

if not hf_token:
    raise RuntimeError("Add HF_TOKEN or HUGGINGFACE_TOKEN to Colab secrets before continuing.")
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token

if kaggle_username and kaggle_key:
    os.environ["KAGGLE_USERNAME"] = kaggle_username
    os.environ["KAGGLE_KEY"] = kaggle_key
else:
    print("Kaggle secrets not found. Validation can still use any local image path you set manually.")

api = HfApi(token=hf_token)
api.create_repo(repo_id=HF_GGUF_REPO_ID, repo_type="model", private=True, exist_ok=True)
print(f"HF GGUF repo ready: {HF_GGUF_REPO_ID}")

## Prepare One X-Ray For llama.cpp Load Validation

This is only for the `llama-mtmd-cli` smoke test. It does not affect conversion.

In [ ]:
run(["python", "download_dataset.py"])
run(["python", "generate_jsonl.py", "--output-dir", "data/processed"])

import json
from pathlib import Path

SAMPLE_IMAGE = None
with open("data/processed/val.jsonl", encoding="utf-8") as handle:
    for line in handle:
        payload = json.loads(line)
        label = payload["messages"][1]["content"].removeprefix("Classification: ")
        if label == "active_tb":
            for item in payload["messages"][0]["content"]:
                if item.get("type") == "image" and item.get("path"):
                    SAMPLE_IMAGE = Path(item["path"])
                    break
        if SAMPLE_IMAGE is not None:
            break

if SAMPLE_IMAGE is None or not SAMPLE_IMAGE.exists():
    raise RuntimeError("Could not find an active_tb validation X-ray for GGUF smoke validation.")
print(SAMPLE_IMAGE)

## Export The Quantized Checkpoint To GGUF

This uses the DRI-23 uploaded LLM Compressor checkpoint as the input source. If this cell fails because llama.cpp cannot read `compressed-tensors` checkpoints directly, use the fallback cell below to export from the merged run #4 checkpoint and quantize into GGUF with llama.cpp. The fallback still creates a deployable Q4_K_M GGUF, but its quantization happens in llama.cpp instead of coming from the LLM Compressor checkpoint.

In [ ]:
run([
    "python", "export_gguf_llamacpp.py",
    "--source", "quantized",
    "--quantized-repo-id", HF_QUANT_REPO_ID,
    "--quantized-dir", str(QUANT_DIR),
    "--output-dir", str(GGUF_DIR),
    "--llama-cpp-dir", str(LLAMA_CPP_DIR),
    "--output-prefix", "drishti-qwen2vl-run4-quantized",
    "--intermediate-outtype", "f16",
    "--quantize-type", "Q4_K_M",
    "--mmproj-outtype", "f16",
    "--validate",
    "--sample-image", str(SAMPLE_IMAGE),
    "--fail-on-validation-error",
], capture=True)

## Fallback: Export From Merged Run #4 Then Quantize In llama.cpp

Run this only if the quantized-source conversion cell fails because current llama.cpp cannot consume the LLM Compressor `compressed-tensors` checkpoint. This is still the standard edge-deployable GGUF path: merged HF model -> f16 GGUF -> Q4_K_M GGUF + f16 mmproj.

In [ ]:
# Optional fallback cell. Leave commented unless direct quantized-source conversion fails.
# if not MERGED_DIR.exists():
#     run([
#         "python", "merge_lora_checkpoint.py",
#         "--output-dir", str(MERGED_DIR),
#         "--torch-dtype", "bfloat16",
#     ], capture=True)
#
# run([
#     "python", "export_gguf_llamacpp.py",
#     "--source", "merged",
#     "--merged-dir", str(MERGED_DIR),
#     "--output-dir", str(GGUF_DIR),
#     "--llama-cpp-dir", str(LLAMA_CPP_DIR),
#     "--output-prefix", "drishti-qwen2vl-run4-merged",
#     "--intermediate-outtype", "f16",
#     "--quantize-type", "Q4_K_M",
#     "--mmproj-outtype", "f16",
#     "--validate",
#     "--sample-image", str(SAMPLE_IMAGE),
# ], capture=True)

## Inspect Export Report

In [ ]:
import json
from pathlib import Path

report_path = GGUF_DIR / "gguf_export_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps({
    "source": report["source"],
    "text_gguf": report["text_gguf"],
    "mmproj_gguf": report["mmproj_gguf"],
    "validation": report["validation"],
}, indent=2))

## Upload GGUF Artifacts To Hugging Face

In [ ]:
api.upload_folder(
    repo_id=HF_GGUF_REPO_ID,
    repo_type="model",
    folder_path=str(GGUF_DIR),
    path_in_repo=".",
    commit_message="Upload DRI-24 Qwen2-VL GGUF export artifacts",
)
print(f"Uploaded to https://huggingface.co/{HF_GGUF_REPO_ID}")